# K-Buffer ATR-Based Breakout Filter - Validation & Grid Search

All-in-one notebook:
- Validate k_buffer logic implementation
- Run grid search to find optimal parameter
- Compare performance metrics

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import math
from pathlib import Path

print("Imports OK")

Imports OK


In [2]:
# Setup paths
nb_path = Path(os.getcwd())
code_path = nb_path.parent / 'code'
data_path = nb_path.parent / 'data' / 'processed' / 'btc_15m.csv'

sys.path.insert(0, str(code_path))

print(f"Code path: {code_path}")
print(f"Data path: {data_path}")
print(f"Files exist: code={code_path.exists()}, data={data_path.exists()}")

Code path: c:\Users\vgvoz\OneDrive\Рабочий стол\breakout-strategy-v2-main\code
Data path: c:\Users\vgvoz\OneDrive\Рабочий стол\breakout-strategy-v2-main\data\processed\btc_15m.csv
Files exist: code=True, data=True


In [3]:
# Import strategy modules
from entry_exit_rules.entry_exit import bos_up, bos_down, SwingLevels, detect_bos_signal
from atr_module.atr_module import get_atr_from_bars

print("Strategy modules imported OK")

Strategy modules imported OK


## Part 1: Validate k_buffer Logic

Test that bos_up() and bos_down() correctly apply buffer parameter

In [4]:
# Test 1: bos_up with buffer
print("="*70)
print("Test 1: bos_up(close, last_high, buffer) function")
print("="*70)

last_high = 100.0
test_cases = [
    (99.5, 100.0, 0.0, False, "No break"),
    (100.5, 100.0, 0.0, True, "Normal break"),
    (100.5, 100.0, 0.3, True, "Break with 0.3 buffer"),
    (100.5, 100.0, 1.0, False, "1.0 buffer too large"),
]

for close, high, buffer, expected, desc in test_cases:
    result = bos_up(close, high, buffer=buffer)
    status = "✓" if result == expected else "✗"
    print(f"{status} close={close:6.1f} high={high:6.1f} buffer={buffer:3.1f} → {result:5} (expect {expected:5}) | {desc}")

print()

Test 1: bos_up(close, last_high, buffer) function
✓ close=  99.5 high= 100.0 buffer=0.0 →     0 (expect     0) | No break
✓ close= 100.5 high= 100.0 buffer=0.0 →     1 (expect     1) | Normal break
✓ close= 100.5 high= 100.0 buffer=0.3 →     1 (expect     1) | Break with 0.3 buffer
✓ close= 100.5 high= 100.0 buffer=1.0 →     0 (expect     0) | 1.0 buffer too large



In [5]:
# Test 2: bos_down with buffer
print("="*70)
print("Test 2: bos_down(close, last_low, buffer) function")
print("="*70)

last_low = 100.0
test_cases = [
    (100.5, 100.0, 0.0, False, "No break"),
    (99.5, 100.0, 0.0, True, "Normal break"),
    (99.5, 100.0, 0.3, True, "Break with 0.3 buffer"),
    (99.5, 100.0, 1.0, False, "1.0 buffer too large"),
]

for close, low, buffer, expected, desc in test_cases:
    result = bos_down(close, low, buffer=buffer)
    status = "✓" if result == expected else "✗"
    print(f"{status} close={close:6.1f} low={low:6.1f} buffer={buffer:3.1f} → {result:5} (expect {expected:5}) | {desc}")

print()

Test 2: bos_down(close, last_low, buffer) function
✓ close= 100.5 low= 100.0 buffer=0.0 →     0 (expect     0) | No break
✓ close=  99.5 low= 100.0 buffer=0.0 →     1 (expect     1) | Normal break
✓ close=  99.5 low= 100.0 buffer=0.3 →     1 (expect     1) | Break with 0.3 buffer
✓ close=  99.5 low= 100.0 buffer=1.0 →     0 (expect     0) | 1.0 buffer too large



In [6]:
# Test 3: Buffer calculation logic (edge cases)
print("="*70)
print("Test 3: Buffer Calculation Logic (k_buffer * atr)")
print("="*70)

test_cases = [
    (0.0, 1.0, 0.0000, "Disabled buffer"),
    (1.0, float('nan'), 0.0000, "NaN ATR"),
    (1.0, None, 0.0000, "None ATR"),
    (1.0, 0.5, 0.5000, "k=1.0, atr=0.5"),
    (1.5, 0.5, 0.7500, "k=1.5, atr=0.5"),
    (2.0, 0.75, 1.5000, "k=2.0, atr=0.75"),
]

for k_buffer, atr, expected_buffer, desc in test_cases:
    buffer = 0.0
    if atr is not None and k_buffer > 0 and not math.isnan(atr):
        buffer = k_buffer * atr
    
    status = "✓" if abs(buffer - expected_buffer) < 0.0001 else "✗"
    print(f"{status} k_buffer={k_buffer:3.1f} atr={str(atr):8s} → buffer={buffer:8.4f} (expect {expected_buffer:8.4f}) | {desc}")

print("\n[OK] All validation tests passed!\n")

Test 3: Buffer Calculation Logic (k_buffer * atr)
✓ k_buffer=0.0 atr=1.0      → buffer=  0.0000 (expect   0.0000) | Disabled buffer
✓ k_buffer=1.0 atr=nan      → buffer=  0.0000 (expect   0.0000) | NaN ATR
✓ k_buffer=1.0 atr=None     → buffer=  0.0000 (expect   0.0000) | None ATR
✓ k_buffer=1.0 atr=0.5      → buffer=  0.5000 (expect   0.5000) | k=1.0, atr=0.5
✓ k_buffer=1.5 atr=0.5      → buffer=  0.7500 (expect   0.7500) | k=1.5, atr=0.5
✓ k_buffer=2.0 atr=0.75     → buffer=  1.5000 (expect   1.5000) | k=2.0, atr=0.75

[OK] All validation tests passed!



## Part 2: Grid Search - Find Optimal k_buffer

Test k_buffer values: 0.0, 0.5, 1.0, 1.5, 2.0

In [7]:
# Load data
print("Loading BTC 15m data...")
df = pd.read_csv(str(data_path))

if 'open_time' in df.columns:
    df = df.rename(columns={'open_time': 'time'})

if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'])

df = df.sort_values(['time'], ascending=True).reset_index(drop=True)

# Use last 2 years for faster iterations
df['year'] = df['time'].dt.year
recent_years = sorted(df['year'].unique())[-2:]
df = df[df['year'].isin(recent_years)].reset_index(drop=True)

print(f"Loaded {len(df):,} bars (last 2 years)")
print(f"Date range: {df['time'].min()} to {df['time'].max()}")
print()

Loading BTC 15m data...
Loaded 42,240 bars (last 2 years)
Date range: 2024-01-01 00:00:00 to 2025-03-15 23:45:00



In [8]:
# Helper functions
def simple_atr(high_low_close, period):
    """Calculate ATR"""
    if len(high_low_close) < period:
        return None
    
    trs = []
    for i in range(len(high_low_close)):
        h, l, c = high_low_close[i]
        cp = high_low_close[i-1][2] if i > 0 else c
        tr = max(h - l, abs(h - cp), abs(l - cp))
        trs.append(tr)
    
    return np.mean(trs[-period:])

def detect_bos_simple(df, t, lookback=20, k_buffer=0.0, atr=None):
    """Simplified BOS detection with k_buffer"""
    if t < lookback:
        return None
    
    current_close = df.loc[t, 'close']
    recent = df.iloc[max(0, t-lookback):t]
    highest_high = recent['high'].max()
    lowest_low = recent['low'].min()
    
    buffer = 0.0
    if atr is not None and k_buffer > 0 and not math.isnan(atr):
        buffer = k_buffer * atr
    
    if current_close > highest_high + buffer:
        return 'LONG'
    elif current_close < lowest_low - buffer:
        return 'SHORT'
    
    return None

print("Helper functions defined OK")

Helper functions defined OK


In [9]:
def run_backtest(df, k_buffer=0.0):
    """Run simplified backtest with given k_buffer"""
    results = {
        'k_buffer': k_buffer,
        'bos_signals': 0,
        'trades': 0,
        'wins': 0,
        'losses': 0,
        'total_pnl': 0.0,
        'total_pnl_pct': 0.0,
        'equity': 100000,
        'trades_pnl': [],
        'max_drawdown': 0.0,
    }
    
    equity = 100000
    peak_equity = 100000
    position = None
    atr_period = 14
    warmup_bars = 50
    
    TP_MULT = 3.0
    FIXED_SL_PCT = 0.01
    RISK_PCT = 0.01
    
    for t in range(warmup_bars, len(df)):
        current_bar = df.iloc[t]
        current_price = float(current_bar['close'])
        high = float(current_bar['high'])
        low = float(current_bar['low'])
        
        # Calculate ATR
        hlc_data = [(float(df.iloc[i]['high']), float(df.iloc[i]['low']), float(df.iloc[i]['close'])) 
                    for i in range(max(0, t-atr_period), t+1)]
        atr = simple_atr(hlc_data, atr_period)
        
        # Handle exit
        if position is not None:
            entry_price = position['price']
            entry_type = position['type']
            qty = position['qty']
            r = entry_price * FIXED_SL_PCT
            
            if entry_type == 'LONG':
                tp_price = entry_price + r * TP_MULT
                sl_price = entry_price - r
                
                if current_price >= tp_price:
                    pnl = qty * (tp_price - entry_price)
                    results['wins'] += 1
                    exit_price = tp_price
                elif low <= sl_price:
                    pnl = qty * (sl_price - entry_price)
                    results['losses'] += 1
                    exit_price = sl_price
                else:
                    pnl = 0
                    exit_price = None
            else:  # SHORT
                tp_price = entry_price - r * TP_MULT
                sl_price = entry_price + r
                
                if current_price <= tp_price:
                    pnl = qty * (entry_price - tp_price)
                    results['wins'] += 1
                    exit_price = tp_price
                elif high >= sl_price:
                    pnl = qty * (entry_price - sl_price)
                    results['losses'] += 1
                    exit_price = sl_price
                else:
                    pnl = 0
                    exit_price = None
            
            if exit_price is not None:
                results['total_pnl'] += pnl
                results['trades_pnl'].append(pnl)
                equity += pnl
                position = None
        
        # Entry logic
        if position is None:
            signal = detect_bos_simple(df, t-1, lookback=20, k_buffer=k_buffer, atr=atr)
            
            if signal is not None:
                results['bos_signals'] += 1
                qty = (equity * RISK_PCT) / (current_price * FIXED_SL_PCT)
                
                if qty > 0:
                    results['trades'] += 1
                    position = {'type': signal, 'price': current_price, 'qty': qty}
        
        # Track drawdown
        if equity > peak_equity:
            peak_equity = equity
        current_dd = (peak_equity - equity) / peak_equity if peak_equity > 0 else 0
        results['max_drawdown'] = max(results['max_drawdown'], current_dd)
    
    # Final calculations
    results['equity'] = equity
    results['total_pnl_pct'] = (equity - 100000) / 100000 * 100
    
    if results['trades'] > 0:
        results['win_rate'] = results['wins'] / results['trades'] * 100
    else:
        results['win_rate'] = 0.0
    
    if len(results['trades_pnl']) > 0:
        pnls = np.array(results['trades_pnl'])
        wins_total = pnls[pnls > 0].sum()
        losses_total = abs(pnls[pnls < 0].sum())
        results['profit_factor'] = wins_total / losses_total if losses_total > 0 else (float('inf') if wins_total > 0 else 0)
    else:
        results['profit_factor'] = 0.0
    
    return results

print("Backtest function defined OK")

Backtest function defined OK


In [10]:
# Run grid search
K_BUFFER_VALUES = [0.0, 0.5, 1.0, 1.5, 2.0]

print("="*70)
print("GRID SEARCH: Testing K_BUFFER Values")
print("="*70)
print()

results_list = []

for k_buf in K_BUFFER_VALUES:
    print(f"Testing k_buffer = {k_buf:.1f}...", end=" ", flush=True)
    result = run_backtest(df, k_buffer=k_buf)
    results_list.append(result)
    print(f"[OK]")
    print(f"  Signals={result['bos_signals']:4d} | Trades={result['trades']:3d} | "
          f"W/L={result['wins']:2d}/{result['losses']:2d} | WR={result['win_rate']:5.1f}% | "
          f"PF={result['profit_factor']:5.2f}")
    print(f"  Return={result['total_pnl_pct']:7.2f}% | MaxDD={result['max_drawdown']*100:6.2f}%")
    print()

print("="*70)

GRID SEARCH: Testing K_BUFFER Values

Testing k_buffer = 0.0... [OK]
  Signals= 717 | Trades=717 | W/L=177/539 | WR= 24.7% | PF= 0.96
  Return= -16.91% | MaxDD= 42.98%

Testing k_buffer = 0.5... [OK]
  Signals= 605 | Trades=605 | W/L=151/453 | WR= 25.0% | PF= 0.98
  Return=  -8.55% | MaxDD= 36.98%

Testing k_buffer = 1.0... [OK]
  Signals= 398 | Trades=398 | W/L=103/295 | WR= 25.9% | PF= 1.03
  Return=   8.30% | MaxDD= 23.00%

Testing k_buffer = 1.5... [OK]
  Signals= 210 | Trades=210 | W/L=64/146 | WR= 30.5% | PF= 1.31
  Return=  52.87% | MaxDD= 18.16%

Testing k_buffer = 2.0... [OK]
  Signals= 104 | Trades=104 | W/L=34/70 | WR= 32.7% | PF= 1.43
  Return=  35.19% | MaxDD= 10.52%



In [11]:
# Create summary table
summary_data = []
for r in results_list:
    summary_data.append({
        'k_buffer': f"{r['k_buffer']:.1f}",
        'Signals': r['bos_signals'],
        'Trades': r['trades'],
        'Wins': r['wins'],
        'Losses': r['losses'],
        'Win_Rate_%': f"{r['win_rate']:.1f}",
        'Profit_Factor': f"{r['profit_factor']:.2f}",
        'Return_%': f"{r['total_pnl_pct']:.2f}",
        'Max_DD_%': f"{r['max_drawdown']*100:.2f}",
    })

summary_df = pd.DataFrame(summary_data)

print("\nSUMMARY TABLE:")
print(summary_df.to_string(index=False))
print()


SUMMARY TABLE:
k_buffer  Signals  Trades  Wins  Losses Win_Rate_% Profit_Factor Return_% Max_DD_%
     0.0      717     717   177     539       24.7          0.96   -16.91    42.98
     0.5      605     605   151     453       25.0          0.98    -8.55    36.98
     1.0      398     398   103     295       25.9          1.03     8.30    23.00
     1.5      210     210    64     146       30.5          1.31    52.87    18.16
     2.0      104     104    34      70       32.7          1.43    35.19    10.52



In [12]:
# Find best results
best_return = max(results_list, key=lambda r: r['total_pnl_pct'])
best_pf = max([r for r in results_list if r['profit_factor'] > 0], 
              key=lambda r: r['profit_factor'], default=None)

print("RECOMMENDATIONS:")
print("-" * 70)
print(f"Best by Return:      k_buffer = {best_return['k_buffer']:.1f} ({best_return['total_pnl_pct']:7.2f}%)")
if best_pf:
    print(f"Best by Profit Factor: k_buffer = {best_pf['k_buffer']:.1f} ({best_pf['profit_factor']:6.2f})")

print()
print("Suggested k_buffer values to try on QuantConnect:")
print(f"  1. Start with: k_buffer = 1.0 (balanced)")
if best_return['k_buffer'] != 1.0:
    print(f"  2. Then optimize to: k_buffer = {best_return['k_buffer']:.1f}")
print()
print("-" * 70)

RECOMMENDATIONS:
----------------------------------------------------------------------
Best by Return:      k_buffer = 1.5 (  52.87%)
Best by Profit Factor: k_buffer = 2.0 (  1.43)

Suggested k_buffer values to try on QuantConnect:
  1. Start with: k_buffer = 1.0 (balanced)
  2. Then optimize to: k_buffer = 1.5

----------------------------------------------------------------------
